# Feature lab — causal discovery, dependence metrics, model effects

Sandbox over the pipeline's LOB features. Reads through `DayFeatureCache` (same featurization as the pipeline); never writes to `data/` or touches pipeline outputs. Results go to `lab/runs/<stamp>_<name>/` (gitignored).

**Flow:** spec → panel → dependence metrics → causal discovery → feature sets → model ablation → (optional) pipeline config for a Kraus run.

Tip: set `source = "synthetic"` first — the synthetic panel has a known causal graph, so you can see what each method gets right before trusting it on market data.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "lab" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np, pandas as pd, matplotlib.pyplot as plt, yaml
from lab import data, metrics, causal, effects, plots, synthetic, bridge
from lab.runs import Experiment
pd.set_option("display.width", 160, "display.max_columns", 30)

## 0 · Spec — every knob for this experiment lives here and is saved with the results

In [ ]:
SPEC = dict(
    name        = "nvda_first_look",
    source      = "config",                 # "config" (market data) | "synthetic"
    config      = "configs/april_nvda.yaml",
    symbol      = None,                     # None = config's symbol
    dates       = 3,                        # int = first n config dates, or a list
    features    = None,                     # None = config predictors + lab EXTRA_FEATURES
    target      = "log_mid_return_fwd_1",   # forward return the models predict
    granger_effect = "log_mid_ret",         # contemporaneous return for Granger/PCMCI
    max_rows    = 400_000,                  # cap for the heavier tests (whole days kept)
    # dependence
    max_lag     = 10,
    # causal discovery
    pc_alpha    = 1e-6,                     # large n → tiny alphas, or read strengths
    pc_max_cond = 3,
    tau_max     = 5,
    pcmci_alpha = 1e-4,
    # transfer entropy (on the pipeline's z-encoded symbols)
    te_features = ["tvi_n", "obi_L1", "ofi_L1_n", "vpin"],
    te_history  = 1,
    te_surrogates = 20,
    # model effects
    model       = "ridge",                  # ridge | logit | hgb | rf | sklearn estimator
    task        = "regression",             # regression | direction (sign → CLS classes)
    lags        = [0],                      # feature lags fed to the model
    n_splits    = 3,                        # walk-forward folds over days
    top_k       = 5,
    n_random    = 3,
)
print(yaml.safe_dump(SPEC, sort_keys=False))
exp = Experiment(SPEC["name"], SPEC)

## 1 · Panel

In [ ]:
if SPEC["source"] == "synthetic":
    panel, truth = synthetic.var_panel(n_days=5, n_per_day=5000)
    FEATS, TARGET, GEFFECT = panel.attrs["features"], "y_fwd_1", "y"
    display(truth)
else:
    panel = data.load(SPEC["config"], dates=SPEC["dates"], symbol=SPEC["symbol"],
                      features=SPEC["features"])
    FEATS = [f for f in data.features_of(panel) if f != SPEC["granger_effect"]]
    TARGET, GEFFECT = SPEC["target"], SPEC["granger_effect"]
panel = data.clean(panel, FEATS + [TARGET, GEFFECT])
print(f"{len(panel):,} rows · {panel.day.nunique()} days · {len(FEATS)} features")
print("features:", FEATS)
panel[FEATS + [TARGET]].describe().T

## 2 · Dependence among features and toward the target

In [ ]:
corr = metrics.corr_matrix(panel, FEATS + [TARGET], method="spearman")
mi   = metrics.mi_matrix(panel, FEATS + [TARGET])
exp.figure("corr", plots.heatmap(corr, "Spearman correlation"))
exp.figure("mi",   plots.heatmap(mi, "Mutual information (nats, kNN)"))
display(metrics.vif(panel, FEATS).to_frame().T.round(1))

In [ ]:
table = metrics.target_table(panel, FEATS, TARGET, granger_effect=GEFFECT,
                             max_lag=SPEC["tau_max"])
exp.table("target_table", table)
table.round(5)

In [ ]:
# lead-lag of the top features vs the contemporaneous return
top = table.MI.nlargest(4).index
exp.figure("xcorr", plots.xcorr_plot({f: metrics.xcorr(panel, f, GEFFECT, SPEC["max_lag"]) for f in top},
                                    f"lead-lag vs {GEFFECT}"))

In [ ]:
# Granger network (conditional = VAR test: helps beyond all others' lags)
G = metrics.granger(panel, FEATS + [GEFFECT], max_lag=SPEC["tau_max"], conditional=True,
                    max_rows=SPEC["max_rows"])
exp.table("granger", G)
exp.figure("granger", plots.heatmap(metrics.to_matrix(G, "delta_r2") * 100,
                                    "conditional Granger ΔR² (%) — row causes column", fmt=".2f"))

### Transfer entropy on the symbol streams the Kraus models see

In [ ]:
if SPEC["source"] == "synthetic":   # quartile symbols per day
    sym = panel[["day"]].copy()
    for c in FEATS + [GEFFECT]:
        sym[c + "_sym"] = panel.groupby("day")[c].transform(lambda s: pd.qcut(s, 4, labels=False))
    sym = sym.dropna().astype({c: int for c in sym.columns if c.endswith("_sym")})
else:   # the pipeline's own z-encoding, on levels exactly as the pipeline encodes them
    sym_src = data.load(SPEC["config"], dates=SPEC["dates"], symbol=SPEC["symbol"],
                        features=SPEC["te_features"] + ["log_mid"], targets=[],
                        stationarize_levels=False, verbose=False)
    sym = data.encode_symbols(sym_src, SPEC["te_features"] + ["log_mid"], SPEC["config"])
te = metrics.transfer_entropy(sym, k=SPEC["te_history"], l=SPEC["te_history"],
                              n_surrogates=SPEC["te_surrogates"])
exp.table("transfer_entropy", te)
exp.figure("te", plots.heatmap(metrics.to_matrix(te, "TE_corrected") * 1000,
                               "transfer entropy, bias-corrected (mbits) — row → column", fmt=".2f"))
te.sort_values("TE_corrected", ascending=False).head(10)

## 3 · Causal discovery

In [ ]:
# PC (contemporaneous CPDAG). The forward-return target is a sink: nothing at t is caused by it.
g = causal.pc(panel, FEATS + [TARGET], alpha=SPEC["pc_alpha"], max_cond=SPEC["pc_max_cond"],
              sinks=[TARGET], max_rows=SPEC["max_rows"])
exp.table("pc_edges", g.edges())
exp.figure("pc_graph", plots.graph(g.edges(), highlight=TARGET, title="PC"))
print("PC parents of target:", g.parents(TARGET, include_undirected=True))
print("Markov blanket     :", g.markov_blanket(TARGET))

In [ ]:
# PCMCI (lagged links X[t-τ] → Y[t]); contemporaneous links are pc()'s job.
# Caveat: lagged-only PCMCI can report X[t-1] → Y when the true link is X[t] → Y
# and X is autocorrelated; tigramite's PCMCI+ (causal.tigramite_pcmci(plus=True)) resolves this.
links = causal.pcmci(panel, FEATS + [GEFFECT], tau_max=SPEC["tau_max"],
                     alpha=SPEC["pcmci_alpha"], max_rows=SPEC["max_rows"])
exp.table("pcmci_links", links)
exp.figure("pcmci_graph", plots.graph(causal.summary_graph(links), "cause", "effect", "val",
                                      directed_col=None, label="lag", highlight=GEFFECT,
                                      title="PCMCI (strongest lag per pair)"))
exp.figure("pcmci_into_target", plots.lag_heatmap(links, GEFFECT))
print("PCMCI parents of", GEFFECT, ":", causal.lagged_parents(links, GEFFECT))

In [ ]:
if SPEC["source"] == "synthetic":   # score against the truth
    found = set(map(tuple, links[links.significant][["cause", "lag", "effect"]].values))
    true  = set(map(tuple, truth[truth.lag > 0][["cause", "lag", "effect"]].values))
    print("recovered:", sorted(found & true)); print("missed:", sorted(true - found))
    print("spurious:", sorted(found - true))

## 4 · Effects on models — do the causally selected features carry the signal?

Same walk-forward folds and model for every set; `random*` sets are size-matched controls.

In [ ]:
SETS = effects.feature_sets(FEATS, pc_graph=g, target=TARGET, pcmci_links=links,
                            pcmci_effect=GEFFECT, table=table, top_k=SPEC["top_k"],
                            n_random=SPEC["n_random"])
for k, v in SETS.items(): print(f"{k:>16}: {v}")
abl = effects.ablate(panel, SETS, TARGET, reference="all", model=SPEC["model"], task=SPEC["task"],
                     lags=SPEC["lags"], n_splits=SPEC["n_splits"])
exp.table("ablation", abl)
METRIC = "r2" if SPEC["task"] == "regression" else "bal_acc"
exp.figure("ablation", plots.ablation_bars(abl, METRIC))
abl.round(5)

In [ ]:
d1 = effects.drop_one(panel, FEATS, TARGET, model=SPEC["model"], task=SPEC["task"],
                      lags=SPEC["lags"], n_splits=SPEC["n_splits"], verbose=False)
imp = effects.permutation_importance(panel, FEATS, TARGET, model=SPEC["model"], task=SPEC["task"],
                                     n_splits=SPEC["n_splits"])
exp.table("drop_one", d1); exp.table("perm_importance", imp)
pd.concat([d1[f"Δ{METRIC}"].drop("all").rename(lambda s: s[1:]), imp.importance], axis=1)\
  .sort_values(f"Δ{METRIC}")

## 5 · (Optional) take a feature set to the real Kraus models

Writes a pipeline `RunConfig` into `lab/runs/configs/` with outputs under `outputs/lab/…`; run it with the pipeline (`python -m pipeline run --config …`, then `train`). Level features are mapped back (`d_log_mid` → `log_mid`). Multivariate training has alphabet n_symbols^(1+len) — a compute-box job past ~3 predictors.

In [ ]:
# cfg_path = bridge.pipeline_config(SPEC["config"], SETS["pcmci_parents"], tag="pcmci_parents",
#                                   multivariate=False)

## 6 · Scratch

In [ ]:
# exp.note("...")
# Experiment.list(); Experiment.open("nvda_first_look")